# Refua Peptide Design Notebook: MDM2-p53 Hotspot

> **Audience:** anyone new to peptide design workflows.  
> **Goal:** generate peptide binder candidates against a clinically relevant oncology target using Refua helper APIs.

## Why this target is practical
- **Target:** human **MDM2** N-terminal pocket (classic inhibitor site for p53 interaction).
- **Complex precedent:** PDB **1YCR** captures MDM2 bound to the p53 transactivation peptide.
- **Peptide druggability signal:** stabilized alpha-helical peptides that disrupt MDM2/MDMX (for example ALRN-6924) have reached clinical testing.

## Workflow
1. Define MDM2 target sequence and the pocket region to prioritize.
2. Use `BinderDesigns.peptide(...)` and `BinderDesigns.disulfide_peptide(...)` helper APIs.
3. Fold one design and inspect `binder_specs` + chain design summary.


In [ ]:
%load_ext refua_notebook
import refua_notebook

refua_notebook.activate()


In [ ]:
from refua import BinderDesigns, Complex, Protein


## Biological setup (plain language)

MDM2 can suppress p53 function by binding a short alpha-helical region of p53.
This notebook designs **new peptide candidates** for that same MDM2 pocket.

| Design input | Choice in this notebook | Why |
|---|---|---|
| Target chain | 1YCR MDM2 chain A sequence (109 aa) | Compact, experimentally solved pocket-bearing domain |
| Binding hotspot | `24..80` on MDM2 | Covers the canonical p53-contacting cleft neighborhood |
| Candidate style | Linear 15-mer + disulfide-constrained peptide template | Common early-stage peptide exploration pattern |


In [ ]:
MDM2_NTERM_1YCR = (
    "SQIPASEQETLVRPKPLLLKLLKSVGAQKDTYTMKEVLFYLGQYIMTKRLYDEKQQHIVYCSNDLLGDLFGVPSFSVKEHRKIYTMIYRNLVVVNQQESSDSGTSVSEN"
)

# p53 peptide observed in 1YCR (useful as a biological reference motif).
P53_TAD_REFERENCE = "SQETFSDLWKLLPEN"

MDM2_P53_POCKET_RANGE = "24..80"


## Build peptide design specs with helper APIs

Refua helper APIs let you create candidate peptide templates quickly:
- `BinderDesigns.peptide(length=15)` for a linear 15-mer search space.
- `BinderDesigns.disulfide_peptide(segment_lengths=(6, 4, 3))` for a constrained cyclic-style template (`6C4C3`).

We fold the **linear** setup below and keep the disulfide version ready for follow-up runs.


In [ ]:
target = Protein(
    MDM2_NTERM_1YCR,
    ids="A",
    binding_types={"binding": MDM2_P53_POCKET_RANGE},
)

linear_peptide = BinderDesigns.peptide(length=15, ids="P")
disulfide_peptide = BinderDesigns.disulfide_peptide(
    segment_lengths=(6, 4, 3),
    ids="Q",
)

linear_design = Complex([target, linear_peptide], name="mdm2_linear_peptide_design")
disulfide_design = Complex([target, disulfide_peptide], name="mdm2_disulfide_peptide_design")


In [ ]:
{
    "linear_helper_spec": linear_peptide.sequence,
    "disulfide_helper_spec": disulfide_peptide.sequence,
    "reference_bound_peptide_1ycr": P53_TAD_REFERENCE,
}


## Run one design pass (linear peptide)

`fold()` will produce a design trace and structure hypothesis for triage.

> Interpretation guardrail: this is a computational prioritization step, not experimental proof of binding or efficacy.


In [ ]:
result_linear = linear_design.fold()

result_linear


In [ ]:
result_linear.binder_specs


In [ ]:
result_linear.chain_design_summary()


## Optional follow-up

Run the constrained version for a second hypothesis set:

```python
result_disulfide = disulfide_design.fold()
result_disulfide.binder_specs
result_disulfide.chain_design_summary()
```


## Science references

- RCSB PDB 1YCR (MDM2-p53 peptide co-crystal): https://www.rcsb.org/structure/1YCR
- Kussie et al., *Science* (1996), MDM2 bound to p53 transactivation peptide: https://pubmed.ncbi.nlm.nih.gov/8875929/
- Saleh et al., *Clinical Cancer Research* (2021), ALRN-6924 phase 1: https://pubmed.ncbi.nlm.nih.gov/34301750/
- Graves et al., *Journal of Medicinal Chemistry* (2023), ALRN-6924 discovery: https://pubmed.ncbi.nlm.nih.gov/37439511/
- FDA preclinical research basics (why experimental validation is required): https://www.fda.gov/patients/drug-development-process/step-2-preclinical-research
